In [ ]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Choose a GPU runtime before continuing."

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

# Safely unmount first in case of stale mounts
try:
    drive.flush_and_unmount()
except Exception:
    pass

# Clean up the directory if it exists and is not empty
mount_path = Path("/content/drive")
if mount_path.exists():
    try:
        shutil.rmtree(mount_path)
    except Exception:
        # If rmtree fails due to active mount, we let drive.mount handle the remount
        pass

drive.mount("/content/drive", force_remount=True)

RUN_ROOT = Path("/content/drive/MyDrive/HydroFed/colab_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [ ]:
!git clone --branch feature/colab-experiments \
  https://github.com/Steven-Yabann/HydroFed.git

%cd /content/HydroFed

!grep -v '^torch==' requirements.txt > /tmp/hydrofed-colab-requirements.txt
!python -m pip install -q -r /tmp/hydrofed-colab-requirements.txt
!python -m pip install -q -e . --no-deps

Cloning into 'HydroFed'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (480/480), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 480 (delta 244), reused 414 (delta 186), pack-reused 0 (from 0)
Receiving objects: 100% (480/480), 126.56 KiB | 2.22 MiB/s, done.
Resolving deltas: 100% (244/244), done.
/content/HydroFed
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for hydrofed (pyproject.toml) ... done


In [ ]:
import sys
import importlib.util

print("Python:", sys.version)

for package in [
    "torch",
    "numpy",
    "pandas",
    "sklearn",
    "flwr",
    "ray",
    "yaml",
    "prometheus_client",
    "requests",
    "pytest",
]:
    print(f"{package:20} installed={importlib.util.find_spec(package) is not None}")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
torch                installed=True
numpy                installed=True
pandas               installed=True
sklearn              installed=True
flwr                 installed=False
ray                  installed=False
yaml                 installed=True
prometheus_client    installed=True
requests             installed=True
pytest               installed=True


In [ ]:
!python -m pip show torch numpy pandas scikit-learn flwr ray PyYAML prometheus-client requests pytest

Name: torch
Version: 2.11.0+cpu
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, fastai, hydrofed, peft, sentence-transformers, timm, torchdata, torchvision
---
Name: numpy
Version: 2.1.3
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2024, NumPy Developers.
 All rights reserved.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are
 met:

     * Redistributions of source code must retain the above copyright
        notice, this list of conditions and the following disclaimer.

     * 

In [ ]:
!python -m pip install -q uv
!uv python install 3.11
!uv venv --python 3.11 /content/hydrofed-venv

In [ ]:
PY = "/content/hydrofed-venv/bin/python"

!uv pip install --python {PY} --no-cache \
  torch==2.2.2 \
  --index-url https://download.pytorch.org/whl/cu121

!grep -v '^torch==' requirements.txt > /tmp/hydrofed-py311-requirements.txt

!uv pip install --python {PY} --no-cache \
  -r /tmp/hydrofed-py311-requirements.txt

!uv pip install --python {PY} -e . --no-deps

In [ ]:
!uv pip install --python {PY} "setuptools<81"

In [ ]:
!{PY} -c "import sys, torch, flwr, ray, pandas; print('Python:', sys.version); print('Torch:', torch.__version__); print('Flower:', flwr.__version__); print('Ray:', ray.__version__); print('Pandas:', pandas.__version__); print('CUDA available:', torch.cuda.is_available()); assert torch.cuda.is_available()"

In [ ]:
PY = "/content/hydrofed-venv/bin/python"

!{PY} -m pytest -q

In [ ]:
!{PY} scripts/run_experiment.py \
  --partitions data/partitions \
  --rounds 1 \
  --epochs 1 \
  --device cuda \
  --seed 42 \
  --output "{RUN_ROOT}/smoke_global.pt" \
  --db "{RUN_ROOT}/smoke.db" \
  --evaluate

In [ ]:
!{PY} scripts/run_baseline.py \
  --partitions data/partitions \
  --output "{RUN_ROOT}/baseline_models_seed42" \
  --db "{RUN_ROOT}/seed42.db" \
  --epochs 100 \
  --device cuda \
  --seed 42 | tee "{RUN_ROOT}/baseline_seed42.log"

In [ ]:
!{PY} scripts/run_experiment.py \
  --partitions data/partitions \
  --rounds 100 \
  --epochs 1 \
  --device cuda \
  --seed 42 \
  --output "{RUN_ROOT}/federated_seed42.pt" \
  --db "{RUN_ROOT}/seed42.db" \
  --evaluate | tee "{RUN_ROOT}/federated_seed42.log"

In [ ]:
!{PY} scripts/run_baseline.py \
  --partitions data/partitions \
  --output "{RUN_ROOT}/baseline_models_seed43" \
  --db "{RUN_ROOT}/seed43.db" \
  --epochs 100 \
  --device cuda \
  --seed 43 | tee "{RUN_ROOT}/baseline_seed43.log"

In [ ]:
!{PY} scripts/run_experiment.py \
  --partitions data/partitions \
  --rounds 100 \
  --epochs 1 \
  --device cuda \
  --seed 43 \
  --output "{RUN_ROOT}/federated_seed43.pt" \
  --db "{RUN_ROOT}/seed43.db" \
  --evaluate | tee "{RUN_ROOT}/federated_seed43.log"

In [ ]:
from pathlib import Path
from hydrofed.data.pipeline import load_client_partition

for path in sorted(Path("data/partitions").glob("plant_*.npz")):
    windows, labels = load_client_partition(path)

    train_end = int(len(labels) * 0.6)
    calibration_end = train_end + int(len(labels) * 0.2)

    train_labels = labels[:train_end]
    calibration_labels = labels[train_end:calibration_end]
    test_labels = labels[calibration_end:]

    print(
        f"{path.stem}: "
        f"train anomalies={train_labels.sum()}/{len(train_labels)}, "
        f"calibration anomalies={calibration_labels.sum()}/{len(calibration_labels)}, "
        f"test anomalies={test_labels.sum()}/{len(test_labels)}"
    )

In [ ]:
%cd /content/HydroFed

!git fetch origin
!git switch feature/balanced-partition-control
!git pull --ff-only origin feature/balanced-partition-control

PYTHON = "/content/hydrofed-venv/bin/python"

!{PYTHON} -m pytest tests/test_data.py -q

!mkdir -p /content/drive/MyDrive/HydroFed/ablation

In [ ]:
!{PYTHON} scripts/prepare_data.py \
  --download \
  --input data/SKAB \
  --output data/partitions_temporal_control \
  --partition-strategy temporal \
  --minimum-anomalies 100

Ensuring SKAB is available under data/SKAB...
Saved 11442 windows across 5 clients using recording-aware temporal partitioning; summary: data/partitions_temporal_control/partition_summary.json


In [ ]:
!{PYTHON} scripts/prepare_data.py \
  --input data/SKAB \
  --output data/partitions_iid_control \
  --partition-strategy iid \
  --minimum-anomalies 100 \
  --seed 42

Saved 11442 windows across 5 clients using balanced IID-like partitioning; summary: data/partitions_iid_control/partition_summary.json


In [ ]:
!{PYTHON} scripts/run_experiment.py \
  --partitions data/partitions_temporal_control \
  --rounds 5 \
  --epochs 5 \
  --evaluate \
  --threshold-percentile 95 \
  --device cuda \
  --seed 42 \
  --db /content/drive/MyDrive/HydroFed/ablation/temporal_seed42.db \
  --output /content/drive/MyDrive/HydroFed/ablation/temporal_seed42_global.pt

/content/hydrofed-venv/lib/python3.11/site-packages/ray/_private/parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Using device: cuda
Started federated run: 9db4ea0f003047afaa2f8263e04d31ae
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-11-01 17:36:56,763	INFO worker.py:1621 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'memory': 7805288448.0, 'node:172.28.0.12': 1.0, 'GPU': 1.0, 'node:__internal_head__': 1.0, 'accelerator_type:T4': 1.0, 'CPU': 2.0, 'object_store_memory': 3902644224.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gp

In [ ]:
!{PYTHON} scripts/run_experiment.py \
  --partitions data/partitions_iid_control \
  --rounds 5 \
  --epochs 5 \
  --evaluate \
  --threshold-percentile 95 \
  --device cuda \
  --seed 42 \
  --db /content/drive/MyDrive/HydroFed/ablation/iid_seed42.db \
  --output /content/drive/MyDrive/HydroFed/ablation/iid_seed42_global.pt

/content/hydrofed-venv/lib/python3.11/site-packages/ray/_private/parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Using device: cuda
Started federated run: 812f506ca3c44f5390a5828a2dae58e2
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-11-01 17:37:27,863	INFO worker.py:1621 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'CPU': 2.0, 'accelerator_type:T4': 1.0, 'memory': 7805657088.0, 'object_store_memory': 3902828544.0, 'GPU': 1.0, 'node:172.28.0.12': 1.0, 'node:__internal_head__': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gp

In [ ]:
!{PYTHON} scripts/run_baseline.py \
  --partitions data/partitions_temporal_control \
  --output /content/drive/MyDrive/HydroFed/ablation/temporal_baselines_seed42 \
  --epochs 25 \
  --db /content/drive/MyDrive/HydroFed/ablation/temporal_seed42.db \
  --device cuda \
  --seed 42

{
  "plant_1": {
    "precision": 0.4987012987012987,
    "recall": 0.9552238805970149,
    "f1": 0.6552901023890785,
    "auc_roc": 0.6231848885653035,
    "threshold": 0.012950991094112397,
    "mean_anomaly_score": 0.03554234612863837,
    "loss": 0.009494475972352319
  },
  "plant_2": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "auc_roc": 0.24077417491153322,
    "threshold": 0.06002597138285637,
    "mean_anomaly_score": 0.05495193443744094,
    "loss": 0.012509434539070382
  },
  "plant_3": {
    "precision": 0.43956043956043955,
    "recall": 0.7407407407407407,
    "f1": 0.5517241379310345,
    "auc_roc": 0.5091171215074723,
    "threshold": 0.16546313613653182,
    "mean_anomaly_score": 0.18427102499314257,
    "loss": 0.009047053658925906
  },
  "plant_4": {
    "precision": 0.35,
    "recall": 0.034482758620689655,
    "f1": 0.06278026905829596,
    "auc_roc": 0.5571864213665763,
    "threshold": 0.010304315807297824,
    "mean_anomaly_score": 0.00747994600

In [ ]:
!{PYTHON} scripts/run_baseline.py \
  --partitions data/partitions_iid_control \
  --output /content/drive/MyDrive/HydroFed/ablation/iid_baselines_seed42 \
  --epochs 25 \
  --db /content/drive/MyDrive/HydroFed/ablation/iid_seed42.db \
  --device cuda \
  --seed 42

{
  "plant_1": {
    "precision": 0.5576923076923077,
    "recall": 0.20422535211267606,
    "f1": 0.29896907216494845,
    "auc_roc": 0.49862265073088363,
    "threshold": 0.013005107454955573,
    "mean_anomaly_score": 0.011681832833730966,
    "loss": 0.009558455999384334
  },
  "plant_2": {
    "precision": 0.5660377358490566,
    "recall": 0.2112676056338028,
    "f1": 0.3076923076923077,
    "auc_roc": 0.4863597991735904,
    "threshold": 0.013934734277427197,
    "mean_anomaly_score": 0.01244320344050011,
    "loss": 0.00961301914760186
  },
  "plant_3": {
    "precision": 0.7608695652173914,
    "recall": 0.23026315789473684,
    "f1": 0.35353535353535354,
    "auc_roc": 0.5287802160123436,
    "threshold": 0.014302308904007075,
    "mean_anomaly_score": 0.012916837069195795,
    "loss": 0.00965594702625069
  },
  "plant_4": {
    "precision": 0.5714285714285714,
    "recall": 0.14184397163120568,
    "f1": 0.22727272727272727,
    "auc_roc": 0.46645702306079667,
    "threshold

In [ ]:
%cd /content/HydroFed

!git fetch origin
!git pull --ff-only origin feature/balanced-partition-control

PYTHON = "/content/hydrofed-venv/bin/python"

In [ ]:
!{PYTHON} scripts/run_experiment.py \
  --partitions data/partitions_temporal_control \
  --rounds 5 \
  --epochs 5 \
  --evaluate \
  --threshold-percentile 95 \
  --device cuda \
  --seed 44 \
  --db /content/drive/MyDrive/HydroFed/ablation/temporal_seed43.db \
  --output /content/drive/MyDrive/HydroFed/ablation/temporal_seed43_global.pt

/content/hydrofed-venv/lib/python3.11/site-packages/ray/_private/parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Using device: cuda
Started federated run: a993b65073a6480c9f3fcf15f118b0e9
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-11-01 21:39:11,082	INFO worker.py:1621 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'memory': 7814182503.0, 'node:__internal_head__': 1.0, 'object_store_memory': 3907091251.0, 'GPU': 1.0, 'node:172.28.0.12': 1.0, 'CPU': 2.0, 'accelerator_type:T4': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gp

In [ ]:
!{PYTHON} scripts/run_experiment.py \
  --partitions data/partitions_iid_control \
  --rounds 5 \
  --epochs 5 \
  --evaluate \
  --threshold-percentile 95 \
  --device cuda \
  --seed 44 \
  --db /content/drive/MyDrive/HydroFed/ablation/iid_seed43.db \
  --output /content/drive/MyDrive/HydroFed/ablation/iid_seed43_global.pt

/content/hydrofed-venv/lib/python3.11/site-packages/ray/_private/parameter.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Using device: cuda
Started federated run: 072040bfc47a4523bc68892313b88774
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-11-01 21:39:40,264	INFO worker.py:1621 -- Started a local Ray instance.
INFO :      Flower VCE: Ray initialized with resources: {'object_store_memory': 3907066675.0, 'node:__internal_head__': 1.0, 'memory': 7814133351.0, 'GPU': 1.0, 'node:172.28.0.12': 1.0, 'CPU': 2.0, 'accelerator_type:T4': 1.0}
INFO :      Optimize your simulation with Flower VCE: https://flower.ai/docs/framework/how-to-run-simulations.html
INFO :      Flower VCE: Resources for each Virtual Client: {'num_cpus': 1, 'num_gp

In [ ]:
import json
import sqlite3

for filename in [
    "/content/drive/MyDrive/HydroFed/ablation/temporal_seed43.db",
    "/content/drive/MyDrive/HydroFed/ablation/iid_seed43.db",
]:
    print(f"\n{filename}")

    with sqlite3.connect(filename) as database:
        rows = database.execute(
            """
            SELECT started_at, config_json
            FROM experiment_runs
            WHERE experiment = 'federated'
            ORDER BY started_at
            """
        ).fetchall()

    for started_at, config_json in rows:
        config = json.loads(config_json)
        print(
            f"started={started_at}, "
            f"seed={config['seed']}, "
            f"rounds={config['rounds']}, "
            f"local_epochs={config['local_epochs']}"
        )


/content/drive/MyDrive/HydroFed/ablation/temporal_seed43.db
started=2026-11-01 21:04:37, seed=43, rounds=5, local_epochs=5
started=2026-11-01 21:39:08, seed=44, rounds=5, local_epochs=5

/content/drive/MyDrive/HydroFed/ablation/iid_seed43.db
started=2026-11-01 21:37:37, seed=43, rounds=5, local_epochs=5
started=2026-11-01 21:39:38, seed=44, rounds=5, local_epochs=5
